# Терең оқыту әдістері негізінде жалған жаңалықтарды анықтау жүйесін әзірлеу (қазақ тілінде)
Курстық жұмыс, «Терең оқыту» пәні. Орындаған: Жарқымбай Заңғар Берікұлы, ЖИТ-33 тобы.

**Мақсат:** BiLSTM нейрондық желісі **қазақ тіліндегі** жаңалық мәтінін оқып, оның шын (REAL) немесе жалған (FAKE) екенін анықтайды.

**Жоспар:**
1. Ағылшынша «Fake or Real News» деректер жиынын жүктеу
2. Теңгерілген ішкі жиынды NLLB-200 машиналық аудармасымен **қазақ тіліне** аудару
3. Қазақша мәтінді тазалау (ә, ғ, қ, ң, ө, ұ, ү, һ, і әріптері сақталады)
4. Салыстыру үшін қарапайым модель: TF-IDF + логистикалық регрессия
5. BiLSTM моделін қазақша мәтінге үйрету және тексеру
6. Өз қазақша сөйлеміңізбен тексеру: `predict()`

> Colab-та іске қосу: **Runtime → Change runtime type → T4 GPU**, содан кейін **Runtime → Run all**. Аударма GPU-да шамамен 5–10 минут алады. Аударылған деректер `kz_news.csv` файлына сақталады, келесі жолы қайта аударылмайды.

## 1. Кітапханаларды қосу

In [ ]:
!pip -q install transformers sentencepiece
import os, re, random, collections
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42  # нәтижелер қайталануы үшін кездейсоқтықты бекітеміз
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Құрылғы:', device)

## 2. Деректер жиынын жүктеу
**Fake or Real News**: 6335 ағылшынша жаңалық (`title`, `text`, `label`). Уақытты үнемдеу үшін **теңгерілген 2000 жаңалық** аламыз (1000 FAKE + 1000 REAL), әр жаңалықтан тақырып пен мәтіннің алғашқы 60 сөзін қалдырамыз.

In [ ]:
N_PER_CLASS, N_WORDS = 1000, 60
URL = 'https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv'
src = pd.read_csv(URL).dropna(subset=['text'])
src = pd.concat([src[src['label']==l].sample(N_PER_CLASS, random_state=SEED) for l in ['FAKE','REAL']]).sample(frac=1, random_state=SEED).reset_index(drop=True)
src['en'] = (src['title'].fillna('') + '. ' + src['text'].str.split().str[:N_WORDS].str.join(' ')).str.replace(r'\s+', ' ', regex=True)
print('Таңдалған жаңалықтар:', len(src)); print(src['label'].value_counts())

## 3. Қазақ тіліне машиналық аударма (NLLB-200)
Meta компаниясының **NLLB-200** (`facebook/nllb-200-distilled-600M`) моделі 200 тілге аударады, оның ішінде қазақ тілі (`kaz_Cyrl`). Нәтиже `kz_news.csv` файлына сақталады. Егер файл бұрыннан бар болса (мысалы, репозиторийден жүктелсе), аударма өткізіліп жіберіледі.

In [ ]:
KZ_CSV = 'kz_news.csv'
if os.path.exists(KZ_CSV):
    df = pd.read_csv(KZ_CSV); print('Дайын қазақша деректер жүктелді:', len(df))
else:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from tqdm.auto import tqdm
    name = 'facebook/nllb-200-distilled-600M'
    tok = AutoTokenizer.from_pretrained(name, src_lang='eng_Latn')
    mt = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float16 if device=='cuda' else torch.float32).to(device).eval()
    kaz_id = tok.convert_tokens_to_ids('kaz_Cyrl')
    texts, out, BS = src['en'].tolist(), [], 32
    for i in tqdm(range(0, len(texts), BS), desc='Аударма'):
        enc = tok(texts[i:i+BS], return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            gen = mt.generate(**enc, forced_bos_token_id=kaz_id, max_new_tokens=160, num_beams=1)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    df = pd.DataFrame({'text_kz': out, 'label': src['label'], 'text_en': src['en']})
    df.to_csv(KZ_CSV, index=False)
    del mt; torch.cuda.empty_cache() if device=='cuda' else None
    print('Аударылды және сақталды:', KZ_CSV)
df['y'] = (df['label'] == 'FAKE').astype(int)   # 1 = жалған, 0 = шын
for t, l in df[['text_kz','label']].head(3).values: print(l, '|', t[:200], '\n')

In [ ]:
lens = df['text_kz'].astype(str).str.split().str.len()
plt.figure(figsize=(7,3.5)); plt.hist(lens, bins=40, color='#4C72B0')
plt.title('Қазақша жаңалықтардың ұзындығы (сөзбен)'); plt.xlabel('сөз саны'); plt.ylabel('жаңалық саны')
plt.tight_layout(); plt.savefig('kz_lengths.png', dpi=150); plt.show()
print('Ұзындық медианасы:', int(lens.median()), 'сөз')

## 4. Қазақша мәтінді тазалау және деректерді бөлу
Кіші әріпке келтіреміз, сілтемелер мен артық таңбаларды алып тастаймыз. Кирилл әріптері, оның ішінде **ә ғ қ ң ө ұ ү һ і**, сақталады. Бөлу: **70% үйрету**, **15% валидация**, **15% тест**.

In [ ]:
def clean(t):
    t = str(t).lower().replace('ё', 'е')
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'[^a-zа-яәғқңөұүһі0-9 ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean'] = df['text_kz'].map(clean)
X_train, X_tmp, y_train, y_tmp = train_test_split(df['clean'], df['y'], test_size=0.30, random_state=SEED, stratify=df['y'])
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print('Үйрету / валидация / тест:', len(X_train), len(X_val), len(X_test))
print('Мысал:', X_train.iloc[0][:150])

## 5. Салыстыруға арналған қарапайым модель: TF-IDF + логистикалық регрессия

In [ ]:
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
Xtr = tfidf.fit_transform(X_train); Xte = tfidf.transform(X_test)
logreg = LogisticRegression(max_iter=2000, C=5).fit(Xtr, y_train)
pred_lr = logreg.predict(Xte)

def report(name, y_true, y_pred):
    r = dict(Модель=name, Accuracy=accuracy_score(y_true,y_pred), Precision=precision_score(y_true,y_pred),
             Recall=recall_score(y_true,y_pred), F1=f1_score(y_true,y_pred))
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in r.items()}); return r
res_lr = report('TF-IDF + LogReg', y_test, pred_lr)

## 6. Нейрондық желіге мәтінді дайындау
Сөздік: үйрету жиынындағы ең жиі 15 000 қазақша сөз, әр сөзге нөмір беріледі. Әр жаңалық **120 сөзге** дейін қысқартылады немесе нөлмен толықтырылады.

In [ ]:
MAX_WORDS, MAX_LEN = 15000, 120
counter = collections.Counter(w for t in X_train for w in t.split())
vocab = {w:i+2 for i,(w,_) in enumerate(counter.most_common(MAX_WORDS-2))}   # 0 = бос орын, 1 = белгісіз сөз
print('Сөздік көлемі:', len(vocab)+2)

def encode(t):
    ids = [vocab.get(w,1) for w in t.split()][:MAX_LEN]
    return ids + [0]*(MAX_LEN-len(ids))

def make_loader(X, y, shuffle):
    xs = torch.tensor([encode(t) for t in X]); ys = torch.tensor(y.values, dtype=torch.float32)
    return DataLoader(TensorDataset(xs, ys), batch_size=32, shuffle=shuffle)

train_dl, val_dl, test_dl = make_loader(X_train,y_train,True), make_loader(X_val,y_val,False), make_loader(X_test,y_test,False)
print('Кодталған жаңалық мысалы:', encode(X_train.iloc[0])[:15], '...')

## 7. BiLSTM моделі
- **Embedding**: сөз нөмірін 128 санды векторға айналдырады.
- **BiLSTM**: мәтінді солдан оңға және оңнан солға оқып, контекстті есте сақтайды.
- **Dropout**: шамадан тыс үйренуден (overfitting) қорғайды.
- **Linear + Sigmoid**: жаңалықтың жалған болу ықтималдығын шығарады (0-ден 1-ге дейін).

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size=MAX_WORDS, emb=128, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden*2, 1)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))                    # (batch, ұзындық, 2*hidden)
        mask = (x != 0).unsqueeze(-1).float()
        h = (h*mask).sum(1) / mask.sum(1).clamp(min=1)   # сөздер бойынша орташа
        return self.fc(self.drop(h)).squeeze(-1)         # логит

model = BiLSTM().to(device)
print(model); print('Параметрлер саны:', sum(p.numel() for p in model.parameters()))

## 8. Үйрету
Бір **эпоха** = үйрету жиынын бір рет толық өту. Қателік функциясы `BCEWithLogitsLoss`, оптимизатор **Adam**. Валидацияда ең жақсы нұсқа сақталады.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss(); opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-5)

def evaluate(dl):
    model.eval(); P, Y, L = [], [], 0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device); out = model(xb)
            L += loss_fn(out, yb).item()*len(yb); P += (torch.sigmoid(out)>0.5).int().tolist(); Y += yb.int().tolist()
    return L/len(Y), accuracy_score(Y,P), P

hist = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[]); best = 0
for epoch in range(1, 11):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    tl, ta, _ = evaluate(train_dl); vl, va, _ = evaluate(val_dl)
    for k,v in zip(hist, (tl,vl,ta,va)): hist[k].append(v)
    if va > best: best = va; torch.save(model.state_dict(), 'best_bilstm_kz.pt')
    print(f'Эпоха {epoch}: loss {tl:.3f}/{vl:.3f}  accuracy {ta:.3f}/{va:.3f}  (үйрету/валидация)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,3.5))
ax[0].plot(hist['train_loss'], 'o-', label='үйрету'); ax[0].plot(hist['val_loss'], 'o-', label='валидация'); ax[0].set_title('Қателік (loss)')
ax[1].plot(hist['train_acc'], 'o-', label='үйрету'); ax[1].plot(hist['val_acc'], 'o-', label='валидация'); ax[1].set_title('Дәлдік (accuracy)')
for a in ax: a.set_xlabel('эпоха'); a.legend()
plt.tight_layout(); plt.savefig('kz_training.png', dpi=150); plt.show()

## 9. Тест деректеріндегі қорытынды тексеру
Модель үйрену кезінде көрмеген қазақша жаңалықтарда тексеріледі.

In [ ]:
model.load_state_dict(torch.load('best_bilstm_kz.pt'))
_, _, pred_lstm = evaluate(test_dl)
res_lstm = report('BiLSTM', y_test, pred_lstm)

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_lstm), display_labels=['ШЫН (REAL)','ЖАЛҒАН (FAKE)']).plot(cmap='Blues')
plt.title('BiLSTM қателер матрицасы (қазақ тілі)'); plt.xlabel('Модель болжамы'); plt.ylabel('Шынайы белгі')
plt.tight_layout(); plt.savefig('kz_confusion.png', dpi=150); plt.show()

table = pd.DataFrame([res_lr, res_lstm]).set_index('Модель').round(4)
table.to_csv('results_kz.csv'); table

## 10. Өз қазақша жаңалығыңызбен тексеру
`predict()` функциясына кез келген қазақша мәтін беріңіз.

In [ ]:
def predict(text):
    model.eval()
    x = torch.tensor([encode(clean(text))]).to(device)
    with torch.no_grad(): p = torch.sigmoid(model(x)).item()
    return f'ЖАЛҒАН (FAKE), ықтималдығы {p:.0%}' if p > 0.5 else f'ШЫН (REAL), ықтималдығы {1-p:.0%}'

print('--- Тест жиынынан мысалдар ---')
for t, y in zip(X_test.iloc[:3], y_test.iloc[:3]):
    print('Шынайы:', 'ЖАЛҒАН' if y else 'ШЫН', '| Модель:', predict(t), '|', t[:90], '...')

print('\n--- Өз сөйлемдеріміз ---')
mine = [
    'Президент дүйсенбі күні Сенатта жаңа заң жобасына қол қойды, деп хабарлады Ақ үйдің баспасөз хатшысы.',
    'ШОК! Дәрігерлер жасырып келген құпия: бұл қарапайым жеміс барлық ауруды бір күнде емдейді!',
    'Мемлекеттік департамент сайлау нәтижелері бойынша ресми мәлімдеме жасады.',
    'Билік бұл ақпаратты сізден жасырады! Бүкіл әлемді жасырын ұйым басқарады, бөлісіңіз!',
]
for s in mine: print(predict(s), '|', s)

## 11. Қорытынды
- Ағылшынша деректер жиыны NLLB-200 машиналық аудармасы арқылы **қазақ тіліне** аударылып, қазақша деректер жиыны (`kz_news.csv`) жасалды.
- BiLSTM моделі тікелей **қазақ тіліндегі мәтінде** үйретіліп, тексерілді. Нәтижелер TF-IDF + логистикалық регрессиямен салыстырылды (жоғарыдағы кесте).
- Шектеулер: деректер машиналық аударма, көлемі шағын (2000 жаңалық, әрқайсысы ~60 сөз); модель фактілерді тексермейді, мәтін стилін үйренеді.
- Даму бағыттары: түпнұсқа қазақша жаңалықтар жинау, көптілді трансформерлер (mBERT, XLM-R, KazBERT), веб-интерфейс.